## CPSC 4970 Homework 5 - Jonathan Braun

In [8]:
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import make_scorer

from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

pd.set_option("display.max_colwidth", 120)

## Load and label the data

In [9]:
def load_headlines(path):
    path = Path(path)
    with path.open("r", encoding="utf-8", errors="ignore") as file:
        return [line.strip() for line in file if line.strip()]

clickbait = load_headlines("clickbait_data")
non_clickbait = load_headlines("non_clickbait_data")

print(f"Clickbait headlines: {len(clickbait):,}")
print(f"Non-clickbait headlines: {len(non_clickbait):,}")
print(f"Total headlines: {len(clickbait) + len(non_clickbait):,}")

Clickbait headlines: 15,999
Non-clickbait headlines: 16,001
Total headlines: 32,000


## Create feature list and target vector

In [10]:
X = clickbait + non_clickbait
y = np.array([1] * len(clickbait) + [0] * len(non_clickbait))

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training examples: {len(X_train):,}")
print(f"Testing examples: {len(X_test):,}")
print(f"Training class balance: {np.bincount(y_train)}")
print(f"Testing class balance: {np.bincount(y_test)}")

Training examples: 25,600
Testing examples: 6,400
Training class balance: [12801 12799]
Testing class balance: [3200 3200]


## Feature representation and cross-validation setup

In [11]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0)
}

tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2)
)

## Model 1: k-nearest neighbors

In [12]:
knn_pipeline = Pipeline([
    ("tfidf", tfidf),
    ("classifier", KNeighborsClassifier())
])

knn_params = {
    "classifier__n_neighbors": [3, 5, 7, 11, 15, 21, 31],
    "classifier__weights": ["uniform", "distance"]
}

knn_grid = GridSearchCV(
    knn_pipeline,
    knn_params,
    cv=cv,
    scoring=scoring,
    refit="f1",
    n_jobs=-1,
    return_train_score=False
)

knn_grid.fit(X_train, y_train)

print("Best kNN parameters:")
print(knn_grid.best_params_)
print(f"Best kNN CV F1 score: {knn_grid.best_score_:.4f}")

Best kNN parameters:
{'classifier__n_neighbors': 3, 'classifier__weights': 'distance'}
Best kNN CV F1 score: 0.7106


## Model 2: Naive Bayes

In [13]:
nb_pipeline = Pipeline([
    ("tfidf", tfidf),
    ("classifier", MultinomialNB())
])

nb_params = {
    "classifier__alpha": [0.01, 0.05, 0.1, 0.5, 1.0, 2.0]
}

nb_grid = GridSearchCV(
    nb_pipeline,
    nb_params,
    cv=cv,
    scoring=scoring,
    refit="f1",
    n_jobs=-1,
    return_train_score=False
)

nb_grid.fit(X_train, y_train)

print("Best Naive Bayes parameters:")
print(nb_grid.best_params_)
print(f"Best Naive Bayes CV F1 score: {nb_grid.best_score_:.4f}")

Best Naive Bayes parameters:
{'classifier__alpha': 1.0}
Best Naive Bayes CV F1 score: 0.9566


## Model 3: Multilayer perceptron

In [14]:
mlp_pipeline = Pipeline([
    ("tfidf", tfidf),
    ("classifier", MLPClassifier(
        max_iter=100,
        early_stopping=True,
        random_state=42
    ))
])

mlp_params = {
    "classifier__hidden_layer_sizes": [
        (25,),
        (50,),
        (100,),
        (25, 25),
        (50, 25),
        (50, 50),
        (100, 50),
        (100, 100),
        (75, 25),
        (100, 50, 25)
    ]
}

mlp_grid = GridSearchCV(
    mlp_pipeline,
    mlp_params,
    cv=cv,
    scoring=scoring,
    refit="f1",
    n_jobs=-1,
    return_train_score=False
)

mlp_grid.fit(X_train, y_train)

print("Best MLP parameters:")
print(mlp_grid.best_params_)
print(f"Best MLP CV F1 score: {mlp_grid.best_score_:.4f}")

Best MLP parameters:
{'classifier__hidden_layer_sizes': (25,)}
Best MLP CV F1 score: 0.9572


## Cross-validation and test-set comparison

After selecting the best version of each model through cross-validation, I compare the models using both cross-validation scores and held-out test-set scores. The cross-validation scores show the mean and standard deviation across the 5 folds, while the test-set scores show how the selected models performed on data that was not used during training or cross-validation.

In [15]:
models = {
    "k-nearest neighbors": knn_grid,
    "Naive Bayes": nb_grid,
    "Multilayer perceptron": mlp_grid
}

results = []

for model_name, grid in models.items():
    best_index = grid.best_index_
    y_pred = grid.predict(X_test)

    results.append({
        "Model": model_name,
        "Best Parameters": grid.best_params_,
        "CV Accuracy": f"{grid.cv_results_['mean_test_accuracy'][best_index]:.4f} +/- {grid.cv_results_['std_test_accuracy'][best_index]:.4f}",
        "CV Precision": f"{grid.cv_results_['mean_test_precision'][best_index]:.4f} +/- {grid.cv_results_['std_test_precision'][best_index]:.4f}",
        "CV Recall": f"{grid.cv_results_['mean_test_recall'][best_index]:.4f} +/- {grid.cv_results_['std_test_recall'][best_index]:.4f}",
        "CV F1": f"{grid.cv_results_['mean_test_f1'][best_index]:.4f} +/- {grid.cv_results_['std_test_f1'][best_index]:.4f}",
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test Precision": precision_score(y_test, y_pred, zero_division=0),
        "Test Recall": recall_score(y_test, y_pred, zero_division=0),
        "Test F1": f1_score(y_test, y_pred, zero_division=0)
    })

results_df = pd.DataFrame(results)
results_df

,Model,Best Parameters,CV Accuracy,CV Precision,CV Recall,CV F1,Test Accuracy,Test Precision,Test Recall,Test F1
0,k-nearest neighbors,"{'classifier__n_neighbors': 3, 'classifier__weights': 'distance'}",0.6048 +/- 0.0077,0.5606 +/- 0.0053,0.9706 +/- 0.0100,0.7106 +/- 0.0039,0.620938,0.571166,0.970625,0.719148
1,Naive Bayes,{'classifier__alpha': 1.0},0.9567 +/- 0.0021,0.9590 +/- 0.0027,0.9542 +/- 0.0023,0.9566 +/- 0.0021,0.959844,0.960275,0.959375,0.959825
2,Multilayer perceptron,"{'classifier__hidden_layer_sizes': (25,)}",0.9575 +/- 0.0015,0.9639 +/- 0.0017,0.9507 +/- 0.0037,0.9572 +/- 0.0016,0.961250,0.967384,0.954688,0.960994


In [16]:
best_model_name = results_df.sort_values("Test F1", ascending=False).iloc[0]["Model"]
best_model_name

'Multilayer perceptron'

## Final Report

For this project, I represented the article headlines using TF-IDF vectorization. I used lowercase text, removed English stop words, limited the vocabulary to 10,000 features, and included both individual words and two-word phrases with `ngram_range=(1, 2)`.

I selected F1 score as the main ranking metric. This is a good metric for this problem because it balances precision and recall. In a clickbait classifier, precision matters because the model should avoid incorrectly labeling normal headlines as clickbait, and recall matters because the model should also catch as many clickbait headlines as possible.

The k-nearest neighbors model performed the worst overall. Its best cross-validation result used `n_neighbors=3` and `weights='distance'`. It had a cross-validation F1 score of 0.7106 +/- 0.0039 and a test F1 score of 0.7191. The model had high recall but low precision, meaning it classified many headlines as clickbait but made many false-positive errors.

The Naive Bayes model performed much better. Its best cross-validation result used `alpha=1.0`. It had a cross-validation F1 score of 0.9566 +/- 0.0021 and a test F1 score of 0.9598.

The multilayer perceptron performed slightly better than Naive Bayes. I tested 10 different hidden-layer configurations, and the best configuration was `hidden_layer_sizes=(25,)`. It had a cross-validation F1 score of 0.9572 +/- 0.0016 and a test F1 score of 0.9610.

Based on the results, the multilayer perceptron was the best model overall because it had the highest cross-validation F1 score and the highest test F1 score. However, Naive Bayes was very close, so it would also be a strong choice, especially if speed and simplicity were important.

A classifier like this could be used as a browser plugin. The plugin could scan the article headlines on a webpage, convert each headline into the same TF-IDF representation used during training, and then classify each headline as clickbait or not clickbait. The plugin could warn the user, visually mark likely clickbait headlines, hide them, or give the user a setting to filter them automatically.